In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

In [3]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

README.md: 0.00B [00:00, ?B/s]

C:\Users\11464\.conda\envs\nlp_lyf\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\11464\.cache\huggingface\hub\datasets--bentrevett--multi30k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

1. 设置 `os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`：
这行代码是把Hugging Face的模型/数据集下载源改成了国内镜像站。
这么做是因为官方源在国内访问很慢，很容易下载失败，换成镜像站能大幅提升下载速度和成功率。

2. `dataset = datasets.load_dataset("bentrevett/multi30k")`：
这行代码是加载 `multi30k` 数据集，这是一个英德双语平行语料库，专门用来做机器翻译任务的。

## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [5]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [6]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [7]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [8]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


In [9]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

### `tokenize_example` 函数作用说明
这个函数的作用，是把数据集中的每一条英德双语样本，统一转换成机器翻译模型能直接使用的格式，具体做了这几件事：

1.  **分词与截断**
    - 用 `en_nlp` 对英语句子 `example["en"]` 分词，用 `de_nlp` 对德语句子 `example["de"]` 分词。
    - 同时把分词结果截断到 `max_length` 指定的长度，避免句子太长影响训练。

2.  **统一小写（可选）**
    - 如果 `lower=True`，会把所有分词结果转成小写，减少词汇表的大小，降低模型的学习难度。

3.  **添加句子边界标记**
    - 在分词结果的开头加上 `sos_token`（句子开始标记，通常是`<sos>`）。
    - 在分词结果的结尾加上 `eos_token`（句子结束标记，通常是`<eos>`）。
    - 这样模型就能清楚知道句子从哪里开始、到哪里结束。

4.  **返回处理结果**
    - 最终返回一个字典，包含处理好的英语分词序列 `en_tokens` 和德语分词序列 `de_tokens`，方便后续批量处理。

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


In [10]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### 关键标记与函数说明
1.  **`<sos>`（Start of Sequence）**
    全称是“句子开始标记”，模型看到这个标记，就知道要开始翻译/生成句子了，相当于给模型一个“启动信号”。

2.  **`<eos>`（End of Sequence）**
    全称是“句子结束标记”，模型看到这个标记，就知道句子已经说完了，可以停止输出，避免生成多余的内容。

3.  **`map()` 函数**
    它的作用是把上面定义好的 `tokenize_example` 分词函数，批量应用到训练、验证、测试数据集的每一条样本上，一次性处理所有数据，不用自己写循环，效率更高，也更不容易出错。

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [12]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [15]:
from torchtext.vocab import build_vocab_from_iterator

min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [16]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [17]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [18]:
en_vocab["the"]

7

In [19]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [20]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [21]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [22]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

在构建词汇表时，我们设置了`min_freq=2`，只有在训练数据中出现次数≥2的词才会被加入词汇表。
- `"crime"`这个词在训练数据中出现次数少于2次，没有被收录进词汇表。
- 当遇到词汇表中不存在的词时，会被自动替换为`<unk>`（未知词标记），避免模型报错，同时统一处理低频词。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


这个`numericalize_example`函数的作用，是把分词后的英德句子，批量转换成模型能直接使用的数字索引：
1.  用`en_vocab.lookup_indices`把英语分词序列`en_tokens`转成数字列表`en_ids`。
2.  用`de_vocab.lookup_indices`把德语分词序列`de_tokens`转成数字列表`de_ids`。
3.  最终返回包含`en_ids`和`de_ids`的字典，为后续模型训练做好准备。

In [25]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [26]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [27]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [28]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [29]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

### `get_collate_fn` 与 `get_data_loader` 函数作用说明

1.  **`get_collate_fn(pad_index)`**
    这个函数的作用是创建一个**批次整理函数 `collate_fn`**，解决不同长度句子的对齐问题：
    - 它接收一个批次的数据，把里面所有样本的英语/德语索引序列分别提取出来。
    - 用 `nn.utils.rnn.pad_sequence` 对这些序列进行**填充（padding）**，让同一个批次里的句子长度都变成一样，填充值为 `<pad>` 对应的索引 `pad_index`。
    - 最终返回一个整理好的批次字典，包含 `en_ids` 和 `de_ids`，方便模型直接处理。

2.  **`get_data_loader(dataset, batch_size, pad_index, shuffle=False)`**
    这个函数的作用是为数据集创建 PyTorch 的 `DataLoader`：
    - 它调用上面的 `get_collate_fn` 来获取批次整理函数。
    - 用 `torch.utils.data.DataLoader` 把数据集包装成可迭代的加载器，设置批次大小、是否打乱数据等参数。
    - 最终返回一个 `DataLoader` 对象，后续可以通过循环来迭代获取批次数据，喂给模型训练。

In [30]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [31]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [32]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

### `Encoder` 类代码说明

#### 1. 输入参数
- `input_dim`：输入词汇表的大小（英语词汇表长度）
- `embedding_dim`：词嵌入层的维度，将单词索引映射成向量
- `hidden_dim`：LSTM隐藏层的维度
- `n_layers`：LSTM的层数，这里设置为2层
- `dropout`：Dropout层的丢弃概率，用于防止过拟合

#### 2. 核心组件
- **`nn.Embedding(input_dim, embedding_dim)`**：词嵌入层，将单词的数字索引转换成固定维度的向量
- **`nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)`**：2层LSTM网络，作为编码器的核心，处理序列并生成上下文表示
- **`nn.Dropout(dropout)`**：Dropout层，对嵌入向量随机丢弃部分信息，增强模型泛化能力

#### 3. `forward` 函数处理流程
1.  **词嵌入与Dropout**：
    输入 `src`（形状：`[序列长度, 批次大小]`）先通过 `self.embedding` 转换成向量，再经过 `self.dropout` 进行随机丢弃，输出 `embedded`（形状：`[序列长度, 批次大小, 嵌入维度]`）。
2.  **LSTM前向传播**：
    `embedded` 输入LSTM，得到 `outputs`（所有时间步的隐藏状态）、`hidden`（最后一层的隐藏状态）和 `cell`（最后一层的细胞状态）。
3.  **返回结果**：
    编码器只返回 `hidden` 和 `cell`，作为解码器的初始隐藏和细胞状态，用于后续翻译生成。

In [33]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

### `Decoder` 类工作流程说明

#### 1. 输入参数
- `output_dim`：目标语言词汇表的大小（德语词汇表长度）
- `embedding_dim`：词嵌入层的维度
- `hidden_dim`：LSTM隐藏层的维度，需与编码器保持一致
- `n_layers`：LSTM的层数，与编码器对齐（2层）
- `dropout`：Dropout层的丢弃概率

#### 2. 核心组件
- **`nn.Embedding(output_dim, embedding_dim)`**：目标语言词嵌入层，将德语单词索引转换成向量
- **`nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)`**：2层LSTM网络，接收编码器传来的上下文状态
- **`nn.Linear(hidden_dim, output_dim)`**：全连接层，将LSTM的隐藏状态映射回目标词汇表维度，生成单词概率分布
- **`nn.Dropout(dropout)`**：Dropout层，用于防止过拟合

#### 3. `forward` 函数处理流程
1.  **输入准备**：
    输入 `input` 是上一步生成的单词（形状：`[batch_size]`），先用 `unsqueeze(0)` 增加一个时间步维度，变成 `[1, batch_size]`。
2.  **词嵌入与Dropout**：
    将输入单词通过嵌入层转换为向量，再经过Dropout层处理，输出 `embedded`（形状：`[1, batch_size, embedding_dim]`）。
3.  **LSTM前向传播**：
    用编码器传来的 `hidden` 和 `cell` 作为初始状态，将 `embedded` 输入LSTM，得到新的 `output`、`hidden` 和 `cell`。
4.  **生成预测**：
    对 `output` 去除时间步维度，通过全连接层 `fc_out` 映射成目标词汇表大小的概率分布 `prediction`。
5.  **返回结果**：
    输出 `prediction`（当前步单词的概率）、更新后的 `hidden` 和 `cell`，供下一步继续生成使用。

In [34]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

### `Seq2Seq` 类代码说明

#### 1. 类的初始化与作用
`Seq2Seq` 是将编码器（Encoder）和解码器（Decoder）组合在一起的完整模型，用于实现机器翻译任务。
- 初始化时接收 `encoder`、`decoder` 和设备 `device`。
- 用 `assert` 语句强制检查编码器和解码器的 `hidden_dim` 和 `n_layers` 必须保持一致，保证模型结构兼容。

---

#### 2. `forward` 函数流程（核心）
1.  **参数准备**
    - `src`：源语言（英语）的输入序列，形状为 `[src_length, batch_size]`。
    - `trg`：目标语言（德语）的真实序列，形状为 `[trg_length, batch_size]`。
    - `teacher_forcing_ratio`：使用“教师强制”机制的概率，控制训练时是否用真实单词作为下一个输入。

2.  **编码器处理**
    将 `src` 输入编码器，得到最终的隐藏状态 `hidden` 和细胞状态 `cell`，作为解码器的初始状态。

3.  **初始化输出存储**
    创建一个全零张量 `outputs`，用于保存解码器每一步的预测结果，形状为 `[trg_length, batch_size, trg_vocab_size]`。

4.  **解码器循环生成**
    - 解码器的第一个输入是目标序列的 `<sos>` 标记，即 `trg[0, :]`。
    - 从第1步到 `trg_length-1` 步循环：
      1.  输入 `input`、编码器传来的 `hidden` 和 `cell`，通过解码器得到当前步的预测 `output`，以及更新后的 `hidden` 和 `cell`。
      2.  将预测结果存入 `outputs` 对应位置。
      3.  根据 `teacher_forcing_ratio` 决定下一步的输入：
          - 如果随机数小于该比例，使用真实的目标词 `trg[t]` 作为下一个输入（教师强制）。
          - 否则，使用模型自己预测的词 `top1` 作为下一个输入。

5.  **返回结果**
    最终返回 `outputs`，包含了所有时间步的预测结果，用于后续计算损失。

---

#### 3. `teacher_forcing` 机制说明
- 作用：训练时以一定概率使用真实的目标词作为解码器的输入，而不是模型自己上一步的预测。
- 好处：帮助模型更快收敛，避免早期错误传播放大；
- 控制：通过 `teacher_forcing_ratio` 调整使用真实输入的比例，比如 `0.75` 表示75%的概率用真实词，25%的概率用模型预测词。

In [35]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [36]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

权重初始化

In [37]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [38]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [39]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [40]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [41]:

def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    # 将模型设置为训练模式
    model.train()
    # 初始化本轮训练损失
    epoch_loss = 0

    # 遍历数据加载器中的每一个批次
    for i, batch in enumerate(data_loader):
        # 获取源语言（德语）和目标语言（英语）的输入序列，并移动到指定设备
        src = batch["de_ids"].to(device)
        trg = batch["en_ids"].to(device)
        # src 形状: [src_length, batch_size]
        # trg 形状: [trg_length, batch_size]

        # 梯度清零，防止上一批次的梯度累积
        optimizer.zero_grad()

        # 前向传播，通过模型得到输出
        output = model(src, trg, teacher_forcing_ratio)
        # output 形状: [trg_length, batch_size, trg_vocab_size]

        # 获取输出的维度（目标词汇表大小）
        output_dim = output.shape[-1]

        # 去掉输出序列的第一个元素（<sos>标记），并展平为二维，方便计算损失
        output = output[1:].view(-1, output_dim)
        # 去掉目标序列的第一个元素，并展平为一维
        trg = trg[1:].view(-1)

        # 计算当前批次的损失
        loss = criterion(output, trg)
        # 反向传播，计算梯度
        loss.backward()

        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # 更新模型参数
        optimizer.step()

        # 累加批次损失
        epoch_loss += loss.item()

    # 返回本轮的平均损失
    return epoch_loss / len(data_loader)


def evaluate_fn(model, data_loader, criterion, device):
    # 将模型设置为评估模式
    model.eval()
    # 初始化本轮验证损失
    epoch_loss = 0

    # 评估时不需要计算梯度
    with torch.no_grad():
        # 遍历数据加载器中的每一个批次
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src 形状: [src_length, batch_size]
            # trg 形状: [trg_length, batch_size]

            # 前向传播，关闭教师强制（使用模型自己的预测作为输入）
            output = model(src, trg, 0)
            # output 形状: [trg_length, batch_size, trg_vocab_size]

            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            # 计算当前批次的损失
            loss = criterion(output, trg)
            epoch_loss += loss.item()

    # 返回本轮的平均损失
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [44]:
import tqdm

# 训练超参数设置
n_epochs = 1  # 因模型训练对计算资源要求较高，此处只设立了一轮训练
clip = 1.0        # 梯度裁剪阈值，防止梯度爆炸
teacher_forcing_ratio = 0.5  # 教师强制比例，控制训练时用真实词作为输入的概率

# 初始化最佳验证损失为无穷大
best_valid_loss = float("inf")

# 开始训练循环，tqdm用来显示进度条
for epoch in tqdm.tqdm(range(n_epochs)):
    # 训练模型，获取本轮训练损失
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    
    # 验证模型，获取本轮验证损失（不更新参数）
    valid_loss = evaluate_fn(model, valid_data_loader, criterion, device)

    # 如果当前验证损失比历史最佳更好，就更新最佳损失并保存模型
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")

    # 打印本轮的训练和验证损失，保留3位小数
    print(f"\tTrain Loss: {train_loss:.3f}")
    print(f"\t Val. Loss: {valid_loss:.3f}")

100%|██████████| 1/1 [00:30<00:00, 30.37s/it]

	Train Loss: 5.038
	 Val. Loss: 5.054


# 模型训练

In [45]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

100%|██████████| 1/1 [00:29<00:00, 29.13s/it]

	Train Loss:   4.430 | Train PPL:  83.970
	Valid Loss:   4.714 | Valid PPL: 111.505


# 模型验证

In [46]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

In [47]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [48]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [49]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [50]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'blue',
 'shirt',
 'is',
 'a',
 'a',
 'a',
 '.',
 '<eos>']